# Monte Carlo Coverage Robustness & Decision Optimization Simulation
========================================================================

This notebook provides the computational validation for the **IS EdTech Framework Research**.
It implements four key mathematical and statistical tests to prove the robustness of the gap findings and the Pareto dominance of the proposed framework $F^*$, **incorporating Multi-Attribute Utility Theory (MAUT) to mitigate design tautology bias** and a **Simulated Proof of Concept (PoC) for rater reliability**:

1. **Observed Coverage Scores**: Calculates baseline scores and complexity costs ($C(f) = \sum_d s(f,d)$).
2. **Monte Carlo Robustness Simulation with Leakage (N=50,000)**: Perturbs partial coverage entries ($s(f,d)=0.5$) with a $Uniform(0.3, 0.7)$ distribution for existing frameworks, and applies $Uniform(0.8, 1.0)$ leakage to $F^*$ to represent real-world implementation friction.
3. **Weighted Sensitivity Analysis with Catastrophic Failure and Tipping Point Mapping (N=100,000)**: Samples dimension weight vectors uniformly from a 5-dimensional Dirichlet distribution and samples resource constraints penalty $\alpha \sim Uniform(0.0, 0.12)$. Utility is calculated as: $Utility(f) = WCS_f \cdot FailureMask - \alpha \cdot C(f)$ where FailureMask drops utility by 90% if D1 (TECH) or D2 (PED) is below 0.4. Also maps the tipping point of $\alpha$ where F* dominance falls below 50%.
4. **Formal Gap Persistence Proof (Linear Programming)**: Mathematically proves (Lemma 1) that no convex combination of existing frameworks can close the High-Stakes Test Specificity (D4) gap.
5. **Decision Optimization Model**: Analyzes 6 distinct institutional priority scenarios to demonstrate that no single existing framework is universally optimal, while $F^*$ remains optimal across all scenarios under reasonable adoption costs.
6. **Simulated Proof of Concept (PoC)**: Validates the practical usability and inter-rater reliability of the proposed operational rubrics by calculating Cohen's Kappa ($\kappa$) on synthetic ratings from two independent expert raters.

## 0. Environment Setup
Run the cell below to install the necessary libraries if you are running in **Google Colab**.

In [ ]:
# If running in Google Colab, uncomment and run this line to install libraries
# !pip install numpy scipy matplotlib

import numpy as np
from scipy.stats import dirichlet
from scipy.optimize import linprog
import json
import os
import matplotlib.pyplot as plt

# Enable inline plotting for Jupyter / Colab
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

## 1. Baseline Data & Observed Coverage Matrix
We define the 12 frameworks, 5 dimensions, and complexity costs ($C(f) = \sum_d s(f,d)$).

In [ ]:
FRAMEWORKS = [
    "F1 Chapelle (2001)",
    "F2 Hubbard (2011)",
    "F3 Leakey (2011)",
    "F4 Colpaert (2004)",
    "F5 Rosell-Aguilar (2017)",
    "F6 Almaiah et al. (2021)",
    "F7 Al-Fraihat et al. (2020)",
    "F8 Scheffel/EFLA (2014)",
    "F9 Park & Jo (2019)",
    "F10 UTAUT/Venkatesh (2003)",
    "F11 TRAM/Kampa (2023)",
    "F12 Essafi/MLLA (2025)",
]

DIMENSIONS = ["D1_TECH", "D2_PED", "D3_INST", "D4_HighStakes", "D6_MultiStakeholder"]

# Coverage matrix S[i][j] = s(framework_i, dimension_j)
COVERAGE_MATRIX = np.array([
    # D1    D2    D3    D4    D6
    [0.0,  1.0,  0.0,  0.5,  0.0],   # F1 Chapelle
    [0.0,  1.0,  0.0,  0.0,  0.0],   # F2 Hubbard
    [0.5,  1.0,  0.0,  0.0,  0.0],   # F3 Leakey
    [0.0,  1.0,  0.0,  0.0,  0.0],   # F4 Colpaert
    [0.5,  0.5,  0.0,  0.0,  0.0],   # F5 Rosell-Aguilar
    [1.0,  0.5,  0.5,  0.0,  0.5],   # F6 Almaiah et al.
    [1.0,  0.5,  0.5,  0.0,  0.5],   # F7 Al-Fraihat et al.
    [0.0,  0.0,  1.0,  0.0,  0.5],   # F8 Scheffel/EFLA
    [0.0,  0.0,  1.0,  0.0,  0.5],   # F9 Park & Jo
    [0.5,  0.0,  0.0,  0.0,  0.0],   # F10 UTAUT
    [0.5,  0.5,  0.0,  0.0,  0.0],   # F11 TRAM
    [0.5,  1.0,  0.0,  0.5,  0.0],   # F12 Essafi/MLLA
])

# Proposed framework F* nominal coverage
F_STAR_NOMINAL = np.array([1.0, 1.0, 1.0, 1.0, 1.0])

COMPLEXITY_COSTS = COVERAGE_MATRIX.sum(axis=1)
COMPLEXITY_STAR = F_STAR_NOMINAL.sum()

print("Observed Coverage Scores & Complexity Costs:")
for i, fw in enumerate(FRAMEWORKS):
    print(f"  {fw:<35} CS = {COMPLEXITY_COSTS[i]:.1f}/5.0 (Complexity = {COMPLEXITY_COSTS[i]:.1f})")
print(f"\n  Proposed Framework F*              CS = {COMPLEXITY_STAR:.1f}/5.0 (Complexity = {COMPLEXITY_STAR:.1f})")

## 2. Monte Carlo Robustness Test with Implementation Leakage
We perturb each partial coverage ($0.5$) score using $Uniform(0.3, 0.7)$ for existing frameworks, and perturb $F^*$ using $Uniform(0.8, 1.0)$ to represent real-world implementation leakage.

In [ ]:
N_MONTE_CARLO = 50_000

# Find partial entry coordinates
partial_mask = (COVERAGE_MATRIX == 0.5)
row_indices, col_indices = np.where(partial_mask)
n_partial = len(row_indices)

# Vectorized Monte Carlo
S_perturbed_all = np.tile(COVERAGE_MATRIX, (N_MONTE_CARLO, 1, 1)).astype(float)
rand_draws = np.random.uniform(0.3, 0.7, size=(N_MONTE_CARLO, n_partial))
S_perturbed_all[:, row_indices, col_indices] = rand_draws

mc_cs_all = S_perturbed_all.sum(axis=2)
mc_max_cs = mc_cs_all.max(axis=1)

# Replicated and perturbed F*
F_star_perturbed = np.random.uniform(0.8, 1.0, size=(N_MONTE_CARLO, 5))
mc_cs_star = F_star_perturbed.sum(axis=1)

# Print probability thresholds
thresholds = [2.5, 3.0, 3.5, 4.0]
print("Probability that maximum existing framework CS is below threshold:")
for t in thresholds:
    p = (mc_max_cs < t).mean()
    print(f"  P(max CS < {t:.1f}) = {p*100:.2f}%")

mean_cs_star = mc_cs_star.mean()
print(f"\nExpected CS for F* (with Leakage): {mean_cs_star:.3f} (95% CI: [{np.percentile(mc_cs_star, 2.5):.3f}, {np.percentile(mc_cs_star, 97.5):.3f}])")

# Gap D4 specific analysis
D4_col = 3
mc_D4_values = S_perturbed_all[:, :, D4_col]
p_D4_exceeds = (mc_D4_values.max(axis=1) > 0.7).mean()
print(f"P(D4 gap persists, D4 > 0.7 never achieved by existing) = {(1-p_D4_exceeds)*100:.2f}%")

## 3. Weighted Sensitivity Analysis with Catastrophic Failure and Tipping Point Mapping
We introduce a **catastrophic failure mask** where if a platform lacks core dimensions ($D1\_TECH$ or $D2\_PED$ coverage is $<0.4$), its utility collapses by 90% (modeled as $0.1 \times WCS$). This represents the natural requirement that platforms must be stable and aligned with test formats to be useful.

We also systematically vary $\alpha$ (complexity penalty) from 0.0 to 0.25 to map the exact **tipping point** where $F^*$ dominance probability falls below 50%.

In [ ]:
N_DIRICHLET = 100_000

# Dirichlet weight vectors
weight_samples = dirichlet.rvs([1] * 5, size=N_DIRICHLET)

# Complexity penalty coefficient alpha
alpha_samples = np.random.uniform(0.0, 0.12, size=N_DIRICHLET)

# F* Implementation leakage
F_star_perturbed_sens = np.random.uniform(0.8, 1.0, size=(N_DIRICHLET, 5))

# Calculate Weighted Coverage Scores (WCS)
wcs_existing = weight_samples @ COVERAGE_MATRIX.T
wcs_star_leakage = (weight_samples * F_star_perturbed_sens).sum(axis=1)

# Apply Catastrophic Failure Mask to existing frameworks
fail_mask_existing = (COVERAGE_MATRIX[:, 0] < 0.4) | (COVERAGE_MATRIX[:, 1] < 0.4)
wcs_existing_final = np.copy(wcs_existing)
for i in range(12):
    if fail_mask_existing[i]:
        wcs_existing_final[:, i] = 0.1 * wcs_existing[:, i]

wcs_star_final = wcs_star_leakage

# Utilities: U(f) = WCS_f - alpha * C(f)
ut_existing = wcs_existing_final - alpha_samples[:, np.newaxis] * COMPLEXITY_COSTS
ut_star = wcs_star_final - alpha_samples * COMPLEXITY_STAR

max_ut_existing = ut_existing.max(axis=1)
margin_arr = ut_star - max_ut_existing
dominance_prob = (margin_arr > 0).mean()

print(f"P(F* strictly dominates existing with Catastrophic Failure) = {dominance_prob*100:.2f}%")
print(f"Mean Utility Margin: {margin_arr.mean():.4f} (Min: {margin_arr.min():.4f})")

# Complexity Tipping Point mapping
alpha_values = np.linspace(0.0, 0.25, 26)
tipping_probs = []
for a in alpha_values:
    ut_star_a = wcs_star_final - a * COMPLEXITY_STAR
    ut_existing_a = wcs_existing_final - a * COMPLEXITY_COSTS
    max_ut_existing_a = ut_existing_a.max(axis=1)
    prob_a = (ut_star_a > max_ut_existing_a).mean()
    tipping_probs.append(prob_a)

tipping_point = 0.25
for val, p in zip(alpha_values, tipping_probs):
    if p < 0.50:
        tipping_point = val
        break
print(f"Complexity tipping threshold alpha (where dominance P < 50%) = {tipping_point:.3f}")

## 4. Formal Gap Persistence Proof (Linear Programming)
**Lemma 1**: No convex combination of existing frameworks can achieve full coverage on dimension D4.

In [ ]:
D4_col = 3
D4_scores = COVERAGE_MATRIX[:, D4_col]
n_fw = len(FRAMEWORKS)

c = -D4_scores
A_eq = np.ones((1, n_fw))
b_eq = np.array([1.0])
bounds = [(0, None)] * n_fw

result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
max_D4_convex = -result.fun

print(f"Maximum possible D4 score from any convex combination of F: {max_D4_convex:.4f}")
print(f"Remaining D4 Gap: {1.0 - max_D4_convex:.4f}")
print("Since this max score is 0.5 < 1.0, the D4 gap is mathematically unclosable without F*.")

print("\nGap persistence across all dimensions:")
for j, dim in enumerate(DIMENSIONS):
    c_d = -COVERAGE_MATRIX[:, j]
    res_d = linprog(c_d, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
    max_d = -res_d.fun
    status = "CLOSED" if max_d >= 1.0 else f"PERSISTENT GAP (max = {max_d:.4f})"
    print(f"  {dim:<20} max convex score = {max_d:.4f} -> {status}")

## 5. Decision Optimization under Specific Priority Scenarios
We evaluate the net utility of frameworks under 6 priority scenarios with scenario-specific cost sensitivity $\alpha$ (incorporating the catastrophic failure mask).

In [ ]:
scenarios = {
    "Technical-First (D1 heavy, low friction)": (np.array([0.40, 0.25, 0.15, 0.10, 0.10]), 0.02),
    "Pedagogy-First (D2 heavy, med friction)":  (np.array([0.15, 0.45, 0.15, 0.15, 0.10]), 0.03),
    "Governance-First (D3 heavy, high friction)": (np.array([0.15, 0.20, 0.40, 0.15, 0.10]), 0.05),
    "Test-Aligned (D4 heavy, low friction)":    (np.array([0.15, 0.25, 0.10, 0.40, 0.10]), 0.02),
    "Inclusive (D6 heavy, med friction)":       (np.array([0.15, 0.20, 0.20, 0.15, 0.30]), 0.04),
    "Balanced Equal Weights (med friction)":    (np.array([0.20, 0.20, 0.20, 0.20, 0.20]), 0.04),
}

scenario_details = {}

print(f"{'Scenario':<42} {'Optimal F':<25} {'U_F*':<10} {'U_best':<10} {'Margin'}")
print("-" * 95)
for name, (w, alpha) in scenarios.items():
    wcs_existing_scen = COVERAGE_MATRIX @ w
    wcs_existing_scen_final = np.copy(wcs_existing_scen)
    for i in range(12):
        if fail_mask_existing[i]:
            wcs_existing_scen_final[i] = 0.1 * wcs_existing_scen[i]
            
    ut_existing_scen = wcs_existing_scen_final - alpha * COMPLEXITY_COSTS
    
    ut_star_expected = 0.9 - alpha * COMPLEXITY_STAR
    best_idx = ut_existing_scen.argmax()
    best_fw = FRAMEWORKS[best_idx]
    best_ut = ut_existing_scen[best_idx]
    
    margin_val = ut_star_expected - best_ut
    scenario_details[name] = {"ut_star": ut_star_expected, "ut_best": best_ut}
    print(f"{name:<42} {best_fw:<25} {ut_star_expected:<10.3f} {best_ut:<10.3f} {margin_val:+.3f}")

## 6. Simulated Proof of Concept (PoC) - Rater Reliability
We evaluate the inter-rater reliability (IRR) of the operational rubrics by calculating Cohen's Kappa ($\kappa$) on simulated ratings from two independent experts scoring Platform A (existing) and Platform B (proposed $F^*$-aligned) across 15 sub-indicators.

In [ ]:
r1 = np.array([1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2])
r2 = np.array([1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2])

n_ratings = len(r1)
po = (r1 == r2).mean()

classes = np.unique(np.concatenate([r1, r2]))
pe = 0.0
for c in classes:
    p1 = (r1 == c).sum() / n_ratings
    p2 = (r2 == c).sum() / n_ratings
    pe += p1 * p2

kappa_score = (po - pe) / (1.0 - pe)
print(f"Observed Agreement (po): {po*100:.2f}%")
print(f"Expected Agreement (pe): {pe*100:.2f}%")
print(f"Cohen's Kappa (kappa): {kappa_score:.4f}")

## 7. Figure Generation and Plotting
Here we generate and display the five publication-quality plots (including the new complexity penalty tipping point plot).

In [ ]:
# Color Palette
PALETTE = {
    "blue_light":  "#e3f2fd",
    "blue_mid":    "#1565c0",
    "blue_dark":   "#0d47a1",
    "green_light": "#e8f5e9",
    "green_mid":   "#2e7d32",
    "green_dark":  "#1b5e20",
    "orange_light":"#fff3e0",
    "orange_mid":  "#e65100",
    "red_dark":    "#b71c1c",
    "grey_text":   "#37474f",
    "white":       "#ffffff",
}

# 1. Histogram plot
plt.figure(figsize=(8, 4.5))
plt.hist(mc_max_cs, bins=25, color=PALETTE["blue_mid"], edgecolor='white', alpha=0.7, density=True, label='Max Existing CS')
plt.hist(mc_cs_star, bins=25, color=PALETTE["green_mid"], edgecolor='white', alpha=0.7, density=True, label='Proposed F* CS (with Leakage)')
plt.axvline(x=2.5, color=PALETTE["red_dark"], linestyle='--', linewidth=1.5, label='Observed Max Existing = 2.5')
plt.axvline(x=mean_cs_star, color=PALETTE["green_dark"], linestyle=':', linewidth=1.5, label=f'E[F*] = {mean_cs_star:.2f}')
plt.title("Distribution of Coverage Scores under Uncertainty (Monte Carlo)", fontweight='bold')
plt.xlabel("Coverage Score / 5.0")
plt.ylabel("Probability Density")
plt.legend(loc='upper left')
plt.show()

# 2. Dominance Margin
plt.figure(figsize=(8, 4.5))
plt.hist(margin_arr, bins=45, color=PALETTE["green_mid"], edgecolor='white', alpha=0.85, density=True)
plt.axvline(x=0.0, color=PALETTE["red_dark"], linestyle='-', linewidth=1.5, label='Dominance Boundary')
plt.title("Utility Dominance Margin", fontweight='bold')
plt.xlabel("Net Utility Dominance Margin ($Utility_{F*} - \max Utility_{existing}$)")
plt.ylabel("Probability Density")
plt.legend(loc='upper right')
plt.show()

# 3. Scenario Comparison
short_names = [name.split(" (")[0] for name in scenarios.keys()]
ut_star_vals = [scenario_details[name]["ut_star"] for name in scenarios.keys()]
best_ut_vals = [scenario_details[name]["ut_best"] for name in scenarios.keys()]
x = np.arange(len(short_names))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, ut_star_vals, width, label='Proposed F*', color=PALETTE["green_mid"], alpha=0.9)
ax.bar(x + width/2, best_ut_vals, width, label='Best Existing', color=PALETTE["blue_mid"], alpha=0.9)
ax.set_ylabel("Net Institutional Utility")
ax.set_title("Institutional Net Utility comparison across Priorities", fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=12, ha='right')
ax.legend(loc='upper right')
plt.show()

# 4. Heatmap
conf_matrix = np.zeros((3, 3), dtype=int)
for r1_val, r2_val in zip(r1, r2):
    conf_matrix[r1_val, r2_val] += 1
plt.figure(figsize=(5.5, 4.5))
plt.imshow(conf_matrix, cmap='YlGn')
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{conf_matrix[i,j]}", ha='center', va='center', fontweight='bold')
plt.xticks(np.arange(3), ["0 (Absent)", "1 (Partial)", "2 (Full)"])
plt.yticks(np.arange(3), ["0 (Absent)", "1 (Partial)", "2 (Full)"])
plt.title("Rater Agreement Heatmap", fontweight='bold')
plt.colorbar()
plt.show()

# 5. Tipping Point Plot
plt.figure(figsize=(8, 4.5))
plt.plot(alpha_values, np.array(tipping_probs) * 100, marker='o', color=PALETTE["blue_mid"], linewidth=2, label='F* Dominance Probability')
plt.axvline(x=tipping_point, color=PALETTE["red_dark"], linestyle='--', label=f'Tipping Point (alpha = {tipping_point:.2f})')
plt.axhline(y=50.0, color='grey', linestyle=':')
plt.title("Complexity Penalty Tipping Point Mapping", fontweight='bold')
plt.xlabel("Complexity Penalty Coefficient (alpha)")
plt.ylabel("Dominance Probability (%)")
plt.legend(loc='lower left')
plt.show()

## 8. Export Results
We save the statistical metrics as a JSON file `simulation_results.json` for validation and LaTeX reference.

In [ ]:
ci_results = {}
for i, fw in enumerate(FRAMEWORKS):
    ci_results[fw] = {
        "mean": float(mc_cs_all[:, i].mean()),
        "ci_lo": float(np.percentile(mc_cs_all[:, i], 2.5)),
        "ci_hi": float(np.percentile(mc_cs_all[:, i], 97.5))
    }
ci_results["F* Proposed Framework"] = {
    "mean": float(mc_cs_star.mean()),
    "ci_lo": float(np.percentile(mc_cs_star, 2.5)),
    "ci_hi": float(np.percentile(mc_cs_star, 97.5))
}

export_data = {
    "monte_carlo_n": N_MONTE_CARLO,
    "p_max_below_3_0": float((mc_max_cs < 3.0).mean()),
    "p_max_below_3_5": float((mc_max_cs < 3.5).mean()),
    "p_max_below_4_0": float((mc_max_cs < 4.0).mean()),
    "p_D4_gap_persists": float(1 - p_D4_exceeds),
    "weighted_sensitivity_n": N_DIRICHLET,
    "dominance_probability": float(dominance_prob),
    "min_dominance_margin": float(margin_arr.min()),
    "mean_dominance_margin": float(margin_arr.mean()),
    "complexity_tipping_point": float(tipping_point),
    "lp_max_D4_convex": float(max_D4_convex),
    "framework_ci": ci_results,
    "poc_agreement": float(po),
    "poc_kappa": float(kappa_score)
}

with open("simulation_results.json", "w") as f:
    json.dump(export_data, f, indent=2)

print("Success! Saved results to 'simulation_results.json'.")